In [9]:
import numpy as np
import tensorflow_datasets as tfds
import tensorflow as tf
import matplotlib.pyplot as plt
import keras
from keras import layers
from keras.applications import EfficientNetB0

IMG_SIZE = 224
BATCH_SIZE = 64

In [10]:
dataset_name = "stanford_dogs"
(ds_train, ds_test), ds_info = tfds.load(
    dataset_name, split=["train", "test"], with_info=True, as_supervised=True
)
NUM_CLASSES = ds_info.features["label"].num_classes

In [11]:
size = (IMG_SIZE, IMG_SIZE)
ds_train = ds_train.map(lambda image, label: (tf.image.resize(image, size), label))
ds_test = ds_test.map(lambda image, label: (tf.image.resize(image, size), label))

In [13]:
img_augmentation_layers = [
    layers.RandomRotation(factor=0.15),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
    layers.RandomFlip(),
    layers.RandomContrast(factor=0.1)
]

def img_augmentation(images):
    for layer in img_augmentation_layers:
        images = layer(images)
    return images

In [14]:
# one-hot / categical encoding
def input_preprocess_train(image, label):
    image = img_augmentation(image)
    label = tf.one_hot(label, NUM_CLASSES)
    return image, label

def input_preprocess_test(image, label):
    label = tf.one_hot(label, NUM_CLASSES)
    return image, label

ds_train = ds_train.map(input_preprocess_train, num_parallel_calls=tf.data.AUTOTUNE)
ds_train = ds_train.batch(batch_size=BATCH_SIZE, drop_remainder=True)
ds_train = ds_train.prefetch(tf.data.AUTOTUNE)

ds_test = ds_test.map(input_preprocess_test, num_parallel_calls=tf.data.AUTOTUNE)
ds_test = ds_test.batch(batch_size=BATCH_SIZE, drop_remainder=True)

In [15]:
import tensorflow as tf
model = tf.keras.models.load_model("./saves/stanford_dogs_effcientnet_b0.keras")

In [16]:
# 1 epoch 10분 정도
epochs = 5
hist = model.fit(ds_train, epochs=epochs, validation_data=ds_test)

Epoch 1/5
187/187 ━━━━━━━━━━━━━━━━━━━━ 767s 4s/step - accuracy: 0.1302 - loss: 3.6227 - val_accuracy: 0.0786 - val_loss: 4.2242
Epoch 2/5
187/187 ━━━━━━━━━━━━━━━━━━━━ 800s 4s/step - accuracy: 0.1424 - loss: 3.5399 - val_accuracy: 0.0835 - val_loss: 4.0561
Epoch 3/5
187/187 ━━━━━━━━━━━━━━━━━━━━ 892s 5s/step - accuracy: 0.1573 - loss: 3.4492 - val_accuracy: 0.1056 - val_loss: 3.9157
Epoch 4/5
187/187 ━━━━━━━━━━━━━━━━━━━━ 923s 5s/step - accuracy: 0.1714 - loss: 3.3628 - val_accuracy: 0.1384 - val_loss: 3.6348
Epoch 5/5
187/187 ━━━━━━━━━━━━━━━━━━━━ 741s 4s/step - accuracy: 0.1873 - loss: 3.2740 - val_accuracy: 0.1449 - val_loss: 3.6332
